# Phase 07G: T1-S1 ONNX Runtime Mobile Export Preflight

Notebook này chuyển Student **T1-S1**, seed **123**, sang nhánh
runtime Android đang được ứng dụng sử dụng: **ONNX Runtime Mobile**.

Phạm vi được khóa:

- tái dựng đúng model từ checkpoint đã chọn tại Phase 07A;
- export FP32 encoder/decoder bằng ONNX exporter `dynamo`, fixed shape;
- giới hạn thời gian riêng cho từng component để không lặp lại lỗi 12 giờ;
- kiểm tra ONNX, ONNX Runtime CPU và numerical parity;
- áp dụng dynamic INT8 cho đúng component đã chọn tại Phase 07F;
- kiểm tra tương thích INT8 trên synthetic cohort;
- chưa đánh giá caption quality của ONNX INT8 và chưa benchmark Android;
- không huấn luyện, không chọn lại checkpoint, không mở Held-out Test.

Đầu vào Kaggle:

1. Phase 07F deployment lock;
2. Phase 07A mobile deployment entry;
3. Phase 07B portable model factory;
4. Phase 02A shared Student pipeline;
5. output chứa checkpoint đã chọn của đúng cặp `T1-S1`;
6. nếu Kaggle thiếu: offline wheels tương thích cho `onnx`, `onnxruntime`
   và `onnxscript`.

Runtime: CPU, Internet Off. Ba notebook 07G có thể chạy song song.

Tài liệu kỹ thuật chính thức:

- PyTorch ONNX exporter:
  https://docs.pytorch.org/docs/stable/onnx.html
- ONNX Runtime Mobile:
  https://onnxruntime.ai/docs/tutorials/mobile/
- ONNX Runtime quantization:
  https://onnxruntime.ai/docs/performance/model-optimizations/quantization.html
- Mobile model usability checker:
  https://onnxruntime.ai/docs/tutorials/mobile/helpers/model-usability-checker.html


## Cell 1 — Authorization, provenance và checkpoint


In [1]:
import csv
import hashlib
import importlib.util
import inspect
import json
import math
import os
import platform
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

STARTED = time.perf_counter()
PAIR = "T1-S1"
PAIR_SLUG = "t1_s1"
MARKER_BASE = "T1_S1"
STUDENT_ID = "S1"
SEED = 123
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
NOTEBOOK_ROOT = (
    INPUT_ROOT / "notebooks"
    if (INPUT_ROOT / "notebooks").is_dir()
    else INPUT_ROOT
)
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    f"/kaggle/working//phase7g_{PAIR_SLUG}_ort_mobile_export_preflight",
))
AUDIT = OUT / "audits"
CONFIG = OUT / "configs"
RESULT = OUT / "results"
EXPORT = OUT / "exports"
LOG = OUT / "logs"
PACKAGE = OUT / "package"
MANIFEST = OUT / "manifests"
for directory in [OUT, AUDIT, CONFIG, RESULT, EXPORT, LOG, PACKAGE, MANIFEST]:
    directory.mkdir(parents=True, exist_ok=True)

torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass
torch.set_grad_enabled(False)
torch.manual_seed(42)
np.random.seed(42)


def now():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def sha256(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def exactly_one(name):
    matches = sorted(NOTEBOOK_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target == root or root in target.parents, (root, target)
    return target


def verify_manifest(marker_path, marker):
    reference = marker["manifest"]["csv"]
    path = resolve_inside(marker_path.parent, reference["relative_path"])
    assert path.is_file() and sha256(path) == reference["sha256"]
    with path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    assert len(rows) == int(marker["manifest_rows"])
    for row in rows:
        target = resolve_inside(marker_path.parent, row["relative_path"])
        assert target.is_file()
        assert target.stat().st_size == int(row["size_bytes"])
        assert sha256(target) == row["sha256"]
    return rows


def verify_phase2_manifest(root, marker):
    reference = marker["manifests"]["csv"]
    path = resolve_inside(root, reference["relative_path"])
    assert sha256(path) == reference["sha256"]
    with path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    assert len(rows) == int(marker["artifact_count"])
    for row in rows:
        target = resolve_inside(root, row["relative_path"])
        assert target.is_file()
        assert target.stat().st_size == int(row["size_bytes"])
        assert sha256(target) == row["sha256"]
    return rows


P7F_PATH = exactly_one(
    "PHASE7F_QUANTIZED_QUALITY_AGGREGATE_DEPLOYMENT_LOCK_COMPLETE.json"
)
AUTH_PATH = exactly_one(
    "PHASE7G_MODERN_ANDROID_RUNTIME_PACKAGING_PREFLIGHT_AUTHORIZED.json"
)
P7A_PATH = exactly_one("PHASE7A_MOBILE_DEPLOYMENT_ENTRY_COMPLETE.json")
P7B_PATH = exactly_one(
    "PHASE7B_PORTABLE_MODEL_FACTORY_EXPORT_COMPLETE.json"
)
P2_PATH = exactly_one("PHASE2_SHARED_TRAINING_BUNDLE_READY.json")
P7F, AUTH, P7A, P7B, P2 = map(
    read_json, [P7F_PATH, AUTH_PATH, P7A_PATH, P7B_PATH, P2_PATH]
)

assert P7F["status"] == "complete"
assert P7F["quality_gates_passed"] == "3 / 3"
assert PAIR in P7F["deployment_pairs_locked"]
assert P7F["modern_android_package_created"] is False
assert P7F["physical_android_benchmark_performed"] is False
assert P7F["test_set_status"] == "sealed" and P7F["test_set_used"] is False
assert AUTH["status"] == "authorized"
assert AUTH["mode"] == "modern_android_runtime_packaging_compatibility_preflight"
assert AUTH["locked_seeds"][PAIR] == SEED
assert AUTH["compatibility_artifacts_are_final_android_packages"] is False
assert AUTH["physical_android_benchmark_authorized"] is False
assert AUTH["new_training"] is False
assert AUTH["checkpoint_reselection"] is False
assert AUTH["test_set_status"] == "sealed" and AUTH["test_set_used"] is False
assert P7A["status"] == "complete"
assert P7B["status"] == "complete"
assert P2["status"] == "ready"

p7f_rows = verify_manifest(P7F_PATH, P7F)
p7a_rows = verify_manifest(P7A_PATH, P7A)
p7b_rows = verify_manifest(P7B_PATH, P7B)
p2_rows = verify_phase2_manifest(P2_PATH.parent, P2)

deployment_registry = read_json(
    P7F_PATH.parent / "results/phase7f_locked_deployment_registry.json"
)
deployment_rows = [
    row for row in deployment_registry["rows"] if row["pair"] == PAIR
]
assert len(deployment_rows) == 1
LOCKED = deployment_rows[0]
assert LOCKED["student_id"] == STUDENT_ID
assert int(LOCKED["seed"]) == SEED
assert LOCKED["quality_gate_passed"] is True
locked_contract_path = resolve_inside(
    P7F_PATH.parent, LOCKED["pair_contract"]["relative_path"]
)
assert sha256(locked_contract_path) == LOCKED["pair_contract"]["sha256"]
LOCKED_CONTRACT = read_json(locked_contract_path)
assert LOCKED_CONTRACT["selection_status"] == "locked"
assert LOCKED_CONTRACT["test_set_status"] == "sealed"
assert LOCKED_CONTRACT["test_set_used"] is False

checkpoint_registry = read_json(
    P7A_PATH.parent / "results/phase7a_selected_checkpoint_registry.json"
)
checkpoint_rows = [
    row for row in checkpoint_registry["rows"] if row["pair"] == PAIR
]
assert len(checkpoint_rows) == 1
CHECKPOINT_ROW = checkpoint_rows[0]
assert int(CHECKPOINT_ROW["seed"]) == SEED
CHECKPOINT_PATH = exactly_one(CHECKPOINT_ROW["checkpoint_file"])
assert CHECKPOINT_PATH.stat().st_size == int(
    CHECKPOINT_ROW["checkpoint_size_bytes"]
)
assert sha256(CHECKPOINT_PATH) == CHECKPOINT_ROW["checkpoint_sha256"]

write_json(AUDIT / "phase7g_cell01_entry.json", {
    "schema_version": "phase7g_pair_cell01_entry_v1",
    "created_at_utc": now(),
    "status": "passed",
    "pair": PAIR,
    "student_id": STUDENT_ID,
    "seed": SEED,
    "strategy": LOCKED["strategy"],
    "phase7f_manifest": f"{len(p7f_rows)} / {P7F['manifest_rows']} verified",
    "phase7a_manifest": f"{len(p7a_rows)} / {P7A['manifest_rows']} verified",
    "phase7b_manifest": f"{len(p7b_rows)} / {P7B['manifest_rows']} verified",
    "phase2_manifest": f"{len(p2_rows)} / {P2['artifact_count']} verified",
    "checkpoint_sha256": sha256(CHECKPOINT_PATH),
    "new_training_started": False,
    "checkpoint_reselection": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07G CELL 1 - ENTRY")
print("Status                  : PASSED")
print("Pair / seed             :", PAIR, "/", SEED)
print("Locked strategy         :", LOCKED["strategy"])
print("Upstream manifests      : 4 / 4 VERIFIED")
print("Checkpoint              : SHA-256 VERIFIED")
print("New training            : NOT STARTED")
print("Held-out Test           : SEALED / NOT USED")
print("Progress                : 1 / 6 code cells")


PHASE 07G CELL 1 - ENTRY
Status                  : PASSED
Pair / seed             : T1-S1 / 123
Locked strategy         : encoder_decoder_dynamic_int8_linear
Upstream manifests      : 4 / 4 VERIFIED
Checkpoint              : SHA-256 VERIFIED
New training            : NOT STARTED
Held-out Test           : SEALED / NOT USED
Progress                : 1 / 6 code cells


## Cell 2 — Modern exporter và ONNX Runtime capability


In [2]:
STARTED = time.perf_counter()
required_modules = ["onnx", "onnxruntime", "onnxscript"]
missing_modules = [
    name for name in required_modules
    if importlib.util.find_spec(name) is None
]
if missing_modules:
    wheel_files = sorted(INPUT_ROOT.rglob("*.whl"))
    required_wheel_prefixes = [
        "onnx-1.23.0-",
        "onnxruntime-1.29.0-",
        "onnxscript-0.7.2-",
    ]
    has_required_wheels = all(
        any(path.name.startswith(prefix) for path in wheel_files)
        for prefix in required_wheel_prefixes
    )
    if not has_required_wheels:
        wheel_archives = sorted(
            INPUT_ROOT.rglob("phase7g_ort_offline_wheels.zip")
        )
        assert len(wheel_archives) <= 1, wheel_archives
        if wheel_archives:
            extracted_wheels = OUT / "offline_wheels"
            shutil.unpack_archive(wheel_archives[0], extracted_wheels)
            wheel_files = sorted(extracted_wheels.rglob("*.whl"))
    wheel_directories = sorted({str(path.parent) for path in wheel_files})
    assert wheel_directories, (
        "Thiếu dependency và không tìm thấy offline wheel bundle. "
        "Gắn dataset phase7g_ort_offline_wheels rồi chạy lại."
    )
    command = [
        sys.executable, "-m", "pip", "install",
        "--no-index",
    ]
    for directory in wheel_directories:
        command.extend(["--find-links", directory])
    command.extend([
        "onnx==1.23.0",
        "onnxruntime==1.29.0",
        "onnxscript==0.7.2",
    ])
    completed = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    install_log = LOG / "offline_dependency_install.log"
    install_log.write_text(completed.stdout or "", encoding="utf-8")
    assert completed.returncode == 0, (
        f"Offline dependency install failed; xem {install_log}"
    )
    importlib.invalidate_caches()
    missing_modules = [
        name for name in required_modules
        if importlib.util.find_spec(name) is None
    ]
assert not missing_modules, (
    "Thiếu dependency cho modern ONNX export: "
    f"{missing_modules}. Gắn offline wheel bundle tương thích Python hiện tại "
    "rồi chạy lại; không bật Internet."
)

import onnx
import onnxruntime as ort
import onnxscript
from onnxruntime.quantization import QuantType, quantize_dynamic

export_signature = inspect.signature(torch.onnx.export)
EXPORT_API_PARAMETERS = set(export_signature.parameters)
assert "dynamo" in EXPORT_API_PARAMETERS
EXPORT_SUPPORTS_FALLBACK = "fallback" in EXPORT_API_PARAMETERS
EXPORT_SUPPORTS_EXTERNAL_DATA = "external_data" in EXPORT_API_PARAMETERS
assert "CPUExecutionProvider" in ort.get_available_providers()

TARGET = {
    "application_runtime": "onnxruntime-android",
    "application_runtime_version": "1.29.0",
    "onnx_opset": 17,
    "exporter": "torch.onnx.export(dynamo=True)",
    "fallback_control": (
        "explicit_false"
        if EXPORT_SUPPORTS_FALLBACK
        else "parameter_unavailable_dynamo_path_only"
    ),
    "external_data_control": (
        "explicit_false"
        if EXPORT_SUPPORTS_EXTERNAL_DATA
        else "parameter_unavailable"
    ),
    "shapes": {
        "image": [1, 3, 224, 224],
        "visual_memory": [1, 49, 256],
        "decoder_input_ids": [1, 31],
        "decoder_attention_mask": [1, 31],
        "logits": [1, 31, 10080],
    },
    "dynamic_shapes": False,
    "physical_android_benchmark": False,
}
write_json(CONFIG / "phase7g_ort_mobile_export_contract.json", {
    "schema_version": "phase7g_ort_mobile_export_contract_v1",
    "created_at_utc": now(),
    "pair": PAIR,
    "student_id": STUDENT_ID,
    "seed": SEED,
    "locked_strategy": LOCKED["strategy"],
    "target": TARGET,
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "onnx": onnx.__version__,
        "onnxruntime": ort.__version__,
        "onnxscript": getattr(onnxscript, "__version__", "unknown"),
        "ort_providers": ort.get_available_providers(),
        "cpu_threads": torch.get_num_threads(),
        "torch_onnx_export_parameters": sorted(EXPORT_API_PARAMETERS),
        "fallback_parameter_available": EXPORT_SUPPORTS_FALLBACK,
        "external_data_parameter_available": EXPORT_SUPPORTS_EXTERNAL_DATA,
    },
    "new_training": False,
    "checkpoint_reselection": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})
write_json(AUDIT / "phase7g_cell02_capability.json", {
    "schema_version": "phase7g_pair_cell02_capability_v1",
    "created_at_utc": now(),
    "status": "passed",
    "required_modules": {name: True for name in required_modules},
    "dynamo_exporter": True,
    "fallback_parameter_available": EXPORT_SUPPORTS_FALLBACK,
    "external_data_parameter_available": EXPORT_SUPPORTS_EXTERNAL_DATA,
    "legacy_exporter_used": False,
    "ort_cpu_execution_provider": True,
    "target_android_runtime": "onnxruntime-android:1.29.0",
    "physical_android_benchmark": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07G CELL 2 - CAPABILITY")
print("Status                  : PASSED")
print("PyTorch / ONNX          :", torch.__version__, "/", onnx.__version__)
print("ONNX Runtime            :", ort.__version__)
print("ONNX Script             :", getattr(onnxscript, "__version__", "unknown"))
print("Exporter                : DYNAMO")
print(
    "Fallback control        :",
    "EXPLICIT FALSE"
    if EXPORT_SUPPORTS_FALLBACK
    else "API PARAMETER UNAVAILABLE",
)
print("Android target          : onnxruntime-android 1.29.0")
print("Held-out Test           : SEALED / NOT USED")
print("Progress                : 2 / 6 code cells")


PHASE 07G CELL 2 - CAPABILITY
Status                  : PASSED
PyTorch / ONNX          : 2.11.0+cpu / 1.23.0
ONNX Runtime            : 1.29.0
ONNX Script             : 0.7.2
Exporter                : DYNAMO
Fallback control        : API PARAMETER UNAVAILABLE
Android target          : onnxruntime-android 1.29.0
Held-out Test           : SEALED / NOT USED
Progress                : 2 / 6 code cells


## Cell 3 — Strict reconstruction và bounded ONNX export


In [3]:
STARTED = time.perf_counter()
P7B_ROOT = P7B_PATH.parent
P2_ROOT = P2_PATH.parent
FACTORY_SOURCE = P7B_ROOT / "package/portable_model_factory.py"
assert FACTORY_SOURCE.is_file()
factory_manifest = {
    row["relative_path"]: row for row in p7b_rows
}["package/portable_model_factory.py"]
assert sha256(FACTORY_SOURCE) == factory_manifest["sha256"]

# Export-only adapter: causal_mask đã được tạo tường minh nên truyền hint
# tgt_is_causal=True để bỏ nhánh tự suy luận mask gây GuardOnDataDependentSymNode.
original_factory_source = FACTORY_SOURCE.read_text(encoding="utf-8")
decoder_call = """        hidden = self.transformer_decoder(
            tgt=target,
            memory=visual_memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=~decoder_attention_mask.bool(),
        )
"""
adapted_decoder_call = """        hidden = self.transformer_decoder(
            tgt=target,
            memory=visual_memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=~decoder_attention_mask.bool(),
            tgt_is_causal=True,
        )
"""
assert original_factory_source.count(decoder_call) == 1
adapted_factory_source = original_factory_source.replace(
    decoder_call, adapted_decoder_call
)
FACTORY_PATH = PACKAGE / "portable_model_factory_export_adapter.py"
FACTORY_PATH.write_text(adapted_factory_source, encoding="utf-8")
assert sha256(FACTORY_PATH) != factory_manifest["sha256"]

ENCODER_CONTRACT_PATH = (
    P2_ROOT / f"configs/encoders/{STUDENT_ID.lower()}_encoder_adapter_contract.json"
)
DECODER_CONTRACT_PATH = (
    P2_ROOT / "configs/decoder/shared_transformer_decoder_contract.json"
)
assert ENCODER_CONTRACT_PATH.is_file()
assert DECODER_CONTRACT_PATH.is_file()
ENCODER_CONTRACT = read_json(ENCODER_CONTRACT_PATH)
DECODER_CONTRACT = read_json(DECODER_CONTRACT_PATH)

reference_factory_spec = importlib.util.spec_from_file_location(
    "phase7g_reference_model_factory", FACTORY_SOURCE
)
reference_factory = importlib.util.module_from_spec(reference_factory_spec)
reference_factory_spec.loader.exec_module(reference_factory)
export_factory_spec = importlib.util.spec_from_file_location(
    "phase7g_export_model_factory", FACTORY_PATH
)
export_factory = importlib.util.module_from_spec(export_factory_spec)
export_factory_spec.loader.exec_module(export_factory)

try:
    checkpoint = torch.load(
        CHECKPOINT_PATH, map_location="cpu", weights_only=True
    )
    CHECKPOINT_LOAD_MODE = "weights_only"
except Exception as error:
    assert sha256(CHECKPOINT_PATH) == CHECKPOINT_ROW["checkpoint_sha256"]
    checkpoint = torch.load(
        CHECKPOINT_PATH, map_location="cpu", weights_only=False
    )
    CHECKPOINT_LOAD_MODE = (
        f"trusted_pickle_after_sha256:{type(error).__name__}"
    )
assert "student_encoder_state" in checkpoint
assert "student_decoder_state" in checkpoint

EAGER_ENCODER = reference_factory.build_encoder(
    STUDENT_ID, ENCODER_CONTRACT
).cpu().eval()
EAGER_DECODER = reference_factory.build_decoder(
    DECODER_CONTRACT
).cpu().eval()
EXPORT_DECODER = export_factory.build_decoder(
    DECODER_CONTRACT
).cpu().eval()
encoder_load = EAGER_ENCODER.load_state_dict(
    checkpoint["student_encoder_state"], strict=True
)
decoder_load = EAGER_DECODER.load_state_dict(
    checkpoint["student_decoder_state"], strict=True
)
adapter_load = EXPORT_DECODER.load_state_dict(
    checkpoint["student_decoder_state"], strict=True
)
assert not encoder_load.missing_keys and not encoder_load.unexpected_keys
assert not decoder_load.missing_keys and not decoder_load.unexpected_keys
assert not adapter_load.missing_keys and not adapter_load.unexpected_keys
del checkpoint

encoder_total, _ = reference_factory.count_unique_parameters(EAGER_ENCODER)
decoder_total, _ = reference_factory.count_unique_parameters(EAGER_DECODER)
assert encoder_total == int(ENCODER_CONTRACT["parameters"]["total"])
assert decoder_total == int(
    DECODER_CONTRACT["parameter_budget"]["actual_total_parameters"]
)

# Chứng minh adapter export không đổi đầu ra decoder gốc.
adapter_errors = []
for active_length in [1, 8, 31]:
    token_ids = torch.zeros((1, 31), dtype=torch.long)
    token_ids[:, 0] = 1
    if active_length > 1:
        lexical = (
            torch.arange(1, active_length, dtype=torch.long) * 37 + 5
        ) % 10076 + 4
        token_ids[:, 1:active_length] = lexical
    token_mask = torch.zeros((1, 31), dtype=torch.bool)
    token_mask[:, :active_length] = True
    visual_memory = torch.linspace(
        -0.5, 0.5, steps=1 * 49 * 256, dtype=torch.float32
    ).reshape(1, 49, 256)
    with torch.inference_mode():
        reference_logits = EAGER_DECODER(
            token_ids, visual_memory, token_mask
        )
        adapter_logits = EXPORT_DECODER(
            token_ids, visual_memory, token_mask
        )
    adapter_errors.append(float(torch.max(torch.abs(
        reference_logits.float() - adapter_logits.float()
    )).item()))
ADAPTER_MAX_ABS_ERROR = max(adapter_errors)
assert ADAPTER_MAX_ABS_ERROR <= 1e-7, ADAPTER_MAX_ABS_ERROR
write_json(AUDIT / "phase7g_decoder_export_adapter_equivalence.json", {
    "schema_version": "phase7g_decoder_export_adapter_equivalence_v1",
    "created_at_utc": now(),
    "status": "passed",
    "repair_reason": "avoid_data_dependent_causal_mask_detection",
    "operation": "set_tgt_is_causal_true_with_explicit_causal_mask",
    "samples": 3,
    "maximum_absolute_error": ADAPTER_MAX_ABS_ERROR,
    "gate": 1e-7,
    "original_factory_sha256": factory_manifest["sha256"],
    "adapted_factory_sha256": sha256(FACTORY_PATH),
    "weights_changed": False,
    "checkpoint_changed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})
del EXPORT_DECODER

WORKER_SOURCE = r"""
import argparse
import importlib.util
import inspect
import json
from pathlib import Path

import torch


parser = argparse.ArgumentParser()
parser.add_argument("--component", choices=["encoder", "decoder"], required=True)
parser.add_argument("--student-id", required=True)
parser.add_argument("--factory", required=True)
parser.add_argument("--encoder-contract", required=True)
parser.add_argument("--decoder-contract", required=True)
parser.add_argument("--checkpoint", required=True)
parser.add_argument("--output", required=True)
parser.add_argument("--artifacts-dir", required=True)
args = parser.parse_args()

torch.set_num_threads(1)
torch.set_grad_enabled(False)
torch.manual_seed(42)

factory_spec = importlib.util.spec_from_file_location(
    "phase7g_export_worker_factory", args.factory
)
factory = importlib.util.module_from_spec(factory_spec)
factory_spec.loader.exec_module(factory)
encoder_contract = json.loads(
    Path(args.encoder_contract).read_text(encoding="utf-8")
)
decoder_contract = json.loads(
    Path(args.decoder_contract).read_text(encoding="utf-8")
)
try:
    checkpoint = torch.load(
        args.checkpoint, map_location="cpu", weights_only=True
    )
except Exception:
    checkpoint = torch.load(
        args.checkpoint, map_location="cpu", weights_only=False
    )

if args.component == "encoder":
    model = factory.build_encoder(args.student_id, encoder_contract).cpu().eval()
    result = model.load_state_dict(
        checkpoint["student_encoder_state"], strict=True
    )
    assert not result.missing_keys and not result.unexpected_keys
    inputs = (
        torch.linspace(
            -1.0, 1.0, steps=1 * 3 * 224 * 224, dtype=torch.float32
        ).reshape(1, 3, 224, 224),
    )
    input_names = ["image"]
    output_names = ["visual_memory"]
else:
    model = factory.build_decoder(decoder_contract).cpu().eval()
    result = model.load_state_dict(
        checkpoint["student_decoder_state"], strict=True
    )
    assert not result.missing_keys and not result.unexpected_keys
    token_ids = torch.zeros((1, 31), dtype=torch.long)
    token_ids[:, 0] = 1
    token_ids[:, 1:8] = torch.tensor([5, 11, 23, 47, 89, 144, 233])
    visual_memory = torch.linspace(
        -0.5, 0.5, steps=1 * 49 * 256, dtype=torch.float32
    ).reshape(1, 49, 256)
    attention_mask = torch.zeros((1, 31), dtype=torch.bool)
    attention_mask[:, :8] = True
    inputs = (token_ids, visual_memory, attention_mask)
    input_names = [
        "decoder_input_ids", "visual_memory", "decoder_attention_mask"
    ]
    output_names = ["logits"]

signature = inspect.signature(torch.onnx.export)
parameters = set(signature.parameters)
assert "dynamo" in parameters
Path(args.output).parent.mkdir(parents=True, exist_ok=True)
Path(args.artifacts_dir).mkdir(parents=True, exist_ok=True)
export_kwargs = {
    "export_params": True,
    "opset_version": 17,
    "input_names": input_names,
    "output_names": output_names,
    "dynamo": True,
}
optional_kwargs = {
    "fallback": False,
    "external_data": False,
    "dynamic_shapes": None,
    "report": True,
    "verify": False,
    "optimize": True,
    "artifacts_dir": args.artifacts_dir,
}
for name, value in optional_kwargs.items():
    if name in parameters:
        export_kwargs[name] = value
print(
    "EXPORT_API",
    json.dumps({
        "available_parameters": sorted(parameters),
        "effective_optional_parameters": sorted(
            name for name in optional_kwargs if name in parameters
        ),
        "fallback_control": (
            "explicit_false"
            if "fallback" in parameters
            else "parameter_unavailable_dynamo_path_only"
        ),
    }),
)
with torch.inference_mode():
    torch.onnx.export(
        model,
        inputs,
        args.output,
        **export_kwargs,
    )
assert Path(args.output).is_file()
print(f"EXPORT_OK component={args.component} bytes={Path(args.output).stat().st_size}")
"""
WORKER_PATH = PACKAGE / "modern_onnx_export_worker.py"
WORKER_PATH.write_text(WORKER_SOURCE.strip() + "\n", encoding="utf-8")
compile(WORKER_SOURCE, "modern_onnx_export_worker.py", "exec")

EXPORT_TIMEOUT_SECONDS = int(os.environ.get(
    "KLCN_ONNX_COMPONENT_TIMEOUT_SECONDS", "3600"
))
FP32_ENCODER_PATH = EXPORT / "encoder_fp32.onnx"
FP32_DECODER_PATH = EXPORT / "decoder_fp32.onnx"


def launch_export(component, output_path):
    command = [
        sys.executable,
        str(WORKER_PATH),
        "--component", component,
        "--student-id", STUDENT_ID,
        "--factory", str(FACTORY_PATH),
        "--encoder-contract", str(ENCODER_CONTRACT_PATH),
        "--decoder-contract", str(DECODER_CONTRACT_PATH),
        "--checkpoint", str(CHECKPOINT_PATH),
        "--output", str(output_path),
        "--artifacts-dir", str(LOG / f"{component}_export_report"),
    ]
    started = time.perf_counter()
    status = "failed"
    return_code = None
    error = None
    log_path = LOG / f"{component}_export.log"
    print(
        f"{PAIR} | {component} export STARTED | "
        f"timeout={EXPORT_TIMEOUT_SECONDS}s",
        flush=True,
    )
    with log_path.open("w", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            command,
            cwd=str(OUT),
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            text=True,
            env={
                **os.environ,
                "OMP_NUM_THREADS": "1",
                "MKL_NUM_THREADS": "1",
            },
        )
        while True:
            return_code = process.poll()
            elapsed = time.perf_counter() - started
            if return_code is not None:
                break
            if elapsed >= EXPORT_TIMEOUT_SECONDS:
                process.kill()
                process.wait()
                return_code = int(process.returncode)
                status = "timeout"
                error = f"timeout_after_{EXPORT_TIMEOUT_SECONDS}_seconds"
                print(
                    f"{PAIR} | {component} export TIMEOUT | "
                    f"elapsed={elapsed:.1f}s",
                    flush=True,
                )
                break
            print(
                f"{PAIR} | {component} export RUNNING | "
                f"elapsed={elapsed:.1f}s",
                flush=True,
            )
            time.sleep(30)
    if status != "timeout":
        status = (
            "exported"
            if return_code == 0 and output_path.is_file()
            else "failed"
        )
        if status == "failed":
            error = f"worker_return_code={return_code}"
        print(
            f"{PAIR} | {component} export {status.upper()} | "
            f"elapsed={time.perf_counter() - started:.1f}s",
            flush=True,
        )
    return {
        "component": component,
        "status": status,
        "return_code": return_code,
        "error": error,
        "elapsed_seconds": time.perf_counter() - started,
        "output_path": (
            output_path.relative_to(OUT).as_posix()
            if output_path.is_file() else None
        ),
        "output_size_bytes": (
            output_path.stat().st_size if output_path.is_file() else None
        ),
        "output_sha256": (
            sha256(output_path) if output_path.is_file() else None
        ),
        "log": log_path.relative_to(OUT).as_posix(),
    }


EXPORT_ROWS = [
    launch_export("encoder", FP32_ENCODER_PATH),
    launch_export("decoder", FP32_DECODER_PATH),
]
EXPORTS_COMPLETE = all(row["status"] == "exported" for row in EXPORT_ROWS)
write_json(RESULT / "phase7g_modern_export_registry.json", {
    "schema_version": "phase7g_modern_export_registry_v1",
    "created_at_utc": now(),
    "pair": PAIR,
    "student_id": STUDENT_ID,
    "seed": SEED,
    "exporter": "dynamo",
    "legacy_fallback": False,
    "opset": 17,
    "fixed_shapes": True,
    "component_timeout_seconds": EXPORT_TIMEOUT_SECONDS,
    "decoder_export_adapter": {
        "operation": "set_tgt_is_causal_true_with_explicit_causal_mask",
        "maximum_absolute_error": ADAPTER_MAX_ABS_ERROR,
        "gate": 1e-7,
        "adapted_factory_sha256": sha256(FACTORY_PATH),
    },
    "rows": EXPORT_ROWS,
    "exports_complete": EXPORTS_COMPLETE,
    "new_training": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})
write_json(AUDIT / "phase7g_cell03_export.json", {
    "schema_version": "phase7g_pair_cell03_export_v1",
    "created_at_utc": now(),
    "status": "passed" if EXPORTS_COMPLETE else "technical_blocked",
    "checkpoint_load_mode": CHECKPOINT_LOAD_MODE,
    "strict_state_dict_load": True,
    "encoder_parameters": encoder_total,
    "decoder_parameters": decoder_total,
    "decoder_export_adapter_equivalent": True,
    "decoder_export_adapter_max_abs_error": ADAPTER_MAX_ABS_ERROR,
    "exports": EXPORT_ROWS,
    "new_training": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07G CELL 3 - MODERN ONNX EXPORT")
print("Status                  :", "PASSED" if EXPORTS_COMPLETE else "TECHNICAL BLOCKED")
for row in EXPORT_ROWS:
    print(
        f"{row['component']:8s}                : "
        f"{row['status'].upper()} | {row['elapsed_seconds']:.1f}s"
    )
print("Decoder export adapter  : EQUIVALENT")
print("Adapter max error       :", f"{ADAPTER_MAX_ABS_ERROR:.3e}")
print("Legacy fallback         : DISABLED")
print("New training            : NOT PERFORMED")
print("Held-out Test           : SEALED / NOT USED")
print("Progress                : 3 / 6 code cells")


T1-S1 | encoder export STARTED | timeout=3600s
T1-S1 | encoder export RUNNING | elapsed=0.0s
T1-S1 | encoder export EXPORTED | elapsed=30.0s
T1-S1 | decoder export STARTED | timeout=3600s
T1-S1 | decoder export RUNNING | elapsed=0.0s
T1-S1 | decoder export EXPORTED | elapsed=30.0s
PHASE 07G CELL 3 - MODERN ONNX EXPORT
Status                  : PASSED
encoder                 : EXPORTED | 30.0s
decoder                 : EXPORTED | 30.0s
Decoder export adapter  : EQUIVALENT
Adapter max error       : 0.000e+00
Legacy fallback         : DISABLED
New training            : NOT PERFORMED
Held-out Test           : SEALED / NOT USED
Progress                : 3 / 6 code cells


## Cell 4 — ONNX checker, ORT FP32 parity và mobile diagnostics


In [4]:
STARTED = time.perf_counter()
FP32_PARITY = None
FP32_PARITY_PASSED = False


def max_abs(left, right):
    return float(np.max(np.abs(
        np.asarray(left, dtype=np.float32)
        - np.asarray(right, dtype=np.float32)
    )))


def top1_agreement(left, right, mask):
    left_ids = np.asarray(left).argmax(axis=-1)
    right_ids = np.asarray(right).argmax(axis=-1)
    active = np.asarray(mask, dtype=bool)
    return float(np.mean(left_ids[active] == right_ids[active]))


if EXPORTS_COMPLETE:
    encoder_model = onnx.load(str(FP32_ENCODER_PATH))
    decoder_model = onnx.load(str(FP32_DECODER_PATH))
    onnx.checker.check_model(encoder_model)
    onnx.checker.check_model(decoder_model)
    encoder_ops = sorted({node.op_type for node in encoder_model.graph.node})
    decoder_ops = sorted({node.op_type for node in decoder_model.graph.node})

    options = ort.SessionOptions()
    options.intra_op_num_threads = 1
    options.inter_op_num_threads = 1
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    providers = ["CPUExecutionProvider"]
    fp32_encoder_session = ort.InferenceSession(
        str(FP32_ENCODER_PATH), sess_options=options, providers=providers
    )
    fp32_decoder_session = ort.InferenceSession(
        str(FP32_DECODER_PATH), sess_options=options, providers=providers
    )

    encoder_errors = []
    decoder_errors = []
    decoder_top1 = []
    for sample_index, active_length in enumerate([1, 8, 31]):
        image = torch.linspace(
            -1.0 + 0.05 * sample_index,
            1.0 - 0.05 * sample_index,
            steps=1 * 3 * 224 * 224,
            dtype=torch.float32,
        ).reshape(1, 3, 224, 224)
        token_ids = torch.zeros((1, 31), dtype=torch.long)
        token_ids[:, 0] = 1
        if active_length > 1:
            lexical = (
                torch.arange(1, active_length, dtype=torch.long) * 37 + 5
            ) % 10076 + 4
            token_ids[:, 1:active_length] = lexical
        token_mask = torch.zeros((1, 31), dtype=torch.bool)
        token_mask[:, :active_length] = True
        with torch.inference_mode():
            eager_memory = EAGER_ENCODER(image).cpu().numpy()
            eager_logits = EAGER_DECODER(
                token_ids,
                torch.from_numpy(eager_memory),
                token_mask,
            ).cpu().numpy()
        ort_memory = fp32_encoder_session.run(
            None, {"image": image.numpy()}
        )[0]
        ort_logits = fp32_decoder_session.run(None, {
            "decoder_input_ids": token_ids.numpy(),
            "visual_memory": eager_memory,
            "decoder_attention_mask": token_mask.numpy(),
        })[0]
        encoder_errors.append(max_abs(eager_memory, ort_memory))
        decoder_errors.append(max_abs(eager_logits, ort_logits))
        decoder_top1.append(
            top1_agreement(eager_logits, ort_logits, token_mask.numpy())
        )

    mobile_checker = {}
    for component, path in [
        ("encoder", FP32_ENCODER_PATH),
        ("decoder", FP32_DECODER_PATH),
    ]:
        try:
            checked = subprocess.run(
                [
                    sys.executable,
                    "-m",
                    "onnxruntime.tools.check_onnx_model_mobile_usability",
                    str(path),
                ],
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                timeout=300,
                check=False,
            )
            checker_log = checked.stdout or ""
            checker_status = (
                "completed" if checked.returncode == 0 else "failed"
            )
        except subprocess.TimeoutExpired as exc:
            checker_log = (exc.stdout or "") + "\n" + (exc.stderr or "")
            checker_status = "timeout"
        checker_path = LOG / f"{component}_mobile_usability.log"
        checker_path.write_text(checker_log, encoding="utf-8")
        mobile_checker[component] = {
            "status": checker_status,
            "log": checker_path.relative_to(OUT).as_posix(),
        }

    FP32_PARITY = {
        "encoder_max_abs_error": max(encoder_errors),
        "decoder_max_abs_error": max(decoder_errors),
        "decoder_top1_agreement_mean": float(np.mean(decoder_top1)),
        "encoder_operators": encoder_ops,
        "decoder_operators": decoder_ops,
        "mobile_usability_checker": mobile_checker,
    }
    FP32_PARITY_PASSED = (
        FP32_PARITY["encoder_max_abs_error"] <= 1e-4
        and FP32_PARITY["decoder_max_abs_error"] <= 1e-4
        and FP32_PARITY["decoder_top1_agreement_mean"] == 1.0
    )

write_json(RESULT / "phase7g_fp32_ort_parity.json", {
    "schema_version": "phase7g_fp32_ort_parity_v1",
    "created_at_utc": now(),
    "pair": PAIR,
    "status": (
        "passed" if FP32_PARITY_PASSED
        else ("failed" if EXPORTS_COMPLETE else "not_run")
    ),
    "cohort": "3 deterministic synthetic samples",
    "thresholds": {
        "encoder_max_abs_error": 1e-4,
        "decoder_max_abs_error": 1e-4,
        "decoder_top1_agreement_mean": 1.0,
    },
    "result": FP32_PARITY,
    "validation_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})
write_json(AUDIT / "phase7g_cell04_fp32_parity.json", {
    "schema_version": "phase7g_pair_cell04_fp32_parity_v1",
    "created_at_utc": now(),
    "status": (
        "passed" if FP32_PARITY_PASSED else "technical_blocked"
    ),
    "exports_complete": EXPORTS_COMPLETE,
    "parity": FP32_PARITY,
    "validation_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07G CELL 4 - FP32 ORT PARITY")
print("Status                  :", "PASSED" if FP32_PARITY_PASSED else "TECHNICAL BLOCKED")
if FP32_PARITY is not None:
    print("Encoder max error       :", f"{FP32_PARITY['encoder_max_abs_error']:.3e}")
    print("Decoder max error       :", f"{FP32_PARITY['decoder_max_abs_error']:.3e}")
    print("Decoder top-1           :", f"{FP32_PARITY['decoder_top1_agreement_mean']:.6f}")
print("Validation images       : NOT READ")
print("Held-out Test           : SEALED / NOT USED")
print("Progress                : 4 / 6 code cells")


PHASE 07G CELL 4 - FP32 ORT PARITY
Status                  : PASSED
Encoder max error       : 1.350e-05
Decoder max error       : 6.676e-06
Decoder top-1           : 1.000000
Validation images       : NOT READ
Held-out Test           : SEALED / NOT USED
Progress                : 4 / 6 code cells


## Cell 5 — ONNX Runtime dynamic INT8 compatibility


In [5]:
STARTED = time.perf_counter()
INT8_ENCODER_PATH = EXPORT / "encoder_ort_candidate.onnx"
INT8_DECODER_PATH = EXPORT / "decoder_ort_candidate.onnx"
ORT_INT8_RESULT = None
ORT_INT8_PREFLIGHT_PASSED = False


def cosine(left, right):
    left_tensor = torch.from_numpy(
        np.asarray(left, dtype=np.float32)
    ).reshape(1, -1)
    right_tensor = torch.from_numpy(
        np.asarray(right, dtype=np.float32)
    ).reshape(1, -1)
    return float(F.cosine_similarity(left_tensor, right_tensor).item())


if FP32_PARITY_PASSED:
    quantize_encoder = LOCKED["strategy"] == (
        "encoder_decoder_dynamic_int8_linear"
    )
    quantize_decoder = LOCKED["strategy"] in {
        "encoder_decoder_dynamic_int8_linear",
        "decoder_dynamic_int8_linear",
    }
    if quantize_encoder:
        quantize_dynamic(
            model_input=str(FP32_ENCODER_PATH),
            model_output=str(INT8_ENCODER_PATH),
            per_channel=True,
            reduce_range=False,
            weight_type=QuantType.QInt8,
            op_types_to_quantize=["MatMul", "Gemm"],
        )
    else:
        shutil.copy2(FP32_ENCODER_PATH, INT8_ENCODER_PATH)
    if quantize_decoder:
        quantize_dynamic(
            model_input=str(FP32_DECODER_PATH),
            model_output=str(INT8_DECODER_PATH),
            per_channel=True,
            reduce_range=False,
            weight_type=QuantType.QInt8,
            op_types_to_quantize=["MatMul", "Gemm"],
        )
    else:
        shutil.copy2(FP32_DECODER_PATH, INT8_DECODER_PATH)

    onnx.checker.check_model(onnx.load(str(INT8_ENCODER_PATH)))
    onnx.checker.check_model(onnx.load(str(INT8_DECODER_PATH)))
    options = ort.SessionOptions()
    options.intra_op_num_threads = 1
    options.inter_op_num_threads = 1
    providers = ["CPUExecutionProvider"]
    int8_encoder_session = ort.InferenceSession(
        str(INT8_ENCODER_PATH), sess_options=options, providers=providers
    )
    int8_decoder_session = ort.InferenceSession(
        str(INT8_DECODER_PATH), sess_options=options, providers=providers
    )

    encoder_cosines = []
    decoder_cosines = []
    decoder_top1 = []
    for sample_index, active_length in enumerate([1, 8, 31]):
        image = np.linspace(
            -1.0 + 0.05 * sample_index,
            1.0 - 0.05 * sample_index,
            num=1 * 3 * 224 * 224,
            dtype=np.float32,
        ).reshape(1, 3, 224, 224)
        token_ids = np.zeros((1, 31), dtype=np.int64)
        token_ids[:, 0] = 1
        if active_length > 1:
            lexical = (
                np.arange(1, active_length, dtype=np.int64) * 37 + 5
            ) % 10076 + 4
            token_ids[:, 1:active_length] = lexical
        token_mask = np.zeros((1, 31), dtype=bool)
        token_mask[:, :active_length] = True
        fp32_memory = fp32_encoder_session.run(
            None, {"image": image}
        )[0]
        int8_memory = int8_encoder_session.run(
            None, {"image": image}
        )[0]
        fp32_logits = fp32_decoder_session.run(None, {
            "decoder_input_ids": token_ids,
            "visual_memory": fp32_memory,
            "decoder_attention_mask": token_mask,
        })[0]
        int8_logits = int8_decoder_session.run(None, {
            "decoder_input_ids": token_ids,
            "visual_memory": int8_memory,
            "decoder_attention_mask": token_mask,
        })[0]
        encoder_cosines.append(cosine(fp32_memory, int8_memory))
        decoder_cosines.append(cosine(fp32_logits, int8_logits))
        decoder_top1.append(
            top1_agreement(fp32_logits, int8_logits, token_mask)
        )

    fp32_bytes = (
        FP32_ENCODER_PATH.stat().st_size + FP32_DECODER_PATH.stat().st_size
    )
    candidate_bytes = (
        INT8_ENCODER_PATH.stat().st_size + INT8_DECODER_PATH.stat().st_size
    )
    package_reduction = 1.0 - candidate_bytes / fp32_bytes
    gates = {
        "encoder_cosine": float(np.mean(encoder_cosines)) >= 0.99,
        "decoder_logits_cosine": float(np.mean(decoder_cosines)) >= 0.99,
        "decoder_top1_agreement": float(np.mean(decoder_top1)) >= 0.95,
        "package_size_reduction": package_reduction > 0.0,
        "finite_and_reload": all(math.isfinite(value) for value in [
            *encoder_cosines, *decoder_cosines, *decoder_top1,
            package_reduction,
        ]),
    }
    ORT_INT8_PREFLIGHT_PASSED = all(gates.values())
    ORT_INT8_RESULT = {
        "pair": PAIR,
        "student_id": STUDENT_ID,
        "seed": SEED,
        "strategy": LOCKED["strategy"],
        "quantization": {
            "tool": "onnxruntime.quantization.quantize_dynamic",
            "weight_type": "QInt8",
            "per_channel": True,
            "reduce_range": False,
            "operator_types": ["MatMul", "Gemm"],
            "encoder_quantized": quantize_encoder,
            "decoder_quantized": quantize_decoder,
        },
        "cohort": "3 deterministic synthetic samples",
        "encoder_cosine_mean": float(np.mean(encoder_cosines)),
        "decoder_logits_cosine_mean": float(np.mean(decoder_cosines)),
        "decoder_top1_agreement_mean": float(np.mean(decoder_top1)),
        "fp32_package_size_bytes": fp32_bytes,
        "candidate_package_size_bytes": candidate_bytes,
        "package_reduction_fraction": package_reduction,
        "gates": gates,
        "preflight_passed": ORT_INT8_PREFLIGHT_PASSED,
        "caption_quality_evaluated": False,
        "physical_android_benchmark_performed": False,
    }

write_json(RESULT / "phase7g_ort_int8_compatibility.json", {
    "schema_version": "phase7g_ort_int8_compatibility_v1",
    "created_at_utc": now(),
    "status": (
        "passed" if ORT_INT8_PREFLIGHT_PASSED
        else ("failed" if FP32_PARITY_PASSED else "not_run")
    ),
    "result": ORT_INT8_RESULT,
    "quality_validation_required": True,
    "physical_android_benchmark_performed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})
write_json(AUDIT / "phase7g_cell05_int8_preflight.json", {
    "schema_version": "phase7g_pair_cell05_int8_preflight_v1",
    "created_at_utc": now(),
    "status": (
        "passed" if ORT_INT8_PREFLIGHT_PASSED else "technical_blocked"
    ),
    "result": ORT_INT8_RESULT,
    "caption_quality_evaluated": False,
    "physical_android_benchmark_performed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07G CELL 5 - ORT INT8 PREFLIGHT")
print("Status                  :", "PASSED" if ORT_INT8_PREFLIGHT_PASSED else "TECHNICAL BLOCKED")
if ORT_INT8_RESULT is not None:
    print("Encoder cosine          :", f"{ORT_INT8_RESULT['encoder_cosine_mean']:.6f}")
    print("Decoder cosine          :", f"{ORT_INT8_RESULT['decoder_logits_cosine_mean']:.6f}")
    print("Decoder top-1           :", f"{ORT_INT8_RESULT['decoder_top1_agreement_mean']:.6f}")
    print("Package reduction       :", f"{ORT_INT8_RESULT['package_reduction_fraction']:.2%}")
print("Caption quality         : NOT YET EVALUATED")
print("Physical Android        : NOT BENCHMARKED")
print("Held-out Test           : SEALED / NOT USED")
print("Progress                : 5 / 6 code cells")


PHASE 07G CELL 5 - ORT INT8 PREFLIGHT
Status                  : PASSED
Encoder cosine          : 0.999866
Decoder cosine          : 0.999893
Decoder top-1           : 0.989247
Package reduction       : 23.01%
Caption quality         : NOT YET EVALUATED
Physical Android        : NOT BENCHMARKED
Held-out Test           : SEALED / NOT USED
Progress                : 5 / 6 code cells


## Cell 6 — Package complete/blocked evidence


In [6]:
STARTED = time.perf_counter()
PHASE7G_PASSED = (
    EXPORTS_COMPLETE and FP32_PARITY_PASSED and ORT_INT8_PREFLIGHT_PASSED
)

if PHASE7G_PASSED:
    candidate_artifacts = {}
    for role, path in [
        ("encoder_fp32", FP32_ENCODER_PATH),
        ("decoder_fp32", FP32_DECODER_PATH),
        ("encoder_candidate", INT8_ENCODER_PATH),
        ("decoder_candidate", INT8_DECODER_PATH),
    ]:
        candidate_artifacts[role] = {
            "relative_path": path.relative_to(OUT).as_posix(),
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        }
    ready = {
        "schema_version": "phase7h_ort_validation_quality_input_ready_v1",
        "created_at_utc": now(),
        "status": "ready",
        "pair": PAIR,
        "student_id": STUDENT_ID,
        "seed": SEED,
        "strategy": LOCKED["strategy"],
        "artifacts": candidate_artifacts,
        "compatibility_result": ORT_INT8_RESULT,
        "required_next": (
            "Full Karpathy Validation quality comparison between ONNX FP32 "
            "and ONNX Runtime INT8 candidate before Android integration."
        ),
        "physical_android_benchmark_authorized": False,
        "new_training": False,
        "checkpoint_reselection": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    }
    write_json(
        OUT / f"PHASE7H_{MARKER_BASE}_ORT_INT8_"
        "VALIDATION_QUALITY_INPUT_READY.json",
        ready,
    )

manifest_rows = []
for directory in [AUDIT, CONFIG, RESULT, EXPORT, LOG, PACKAGE]:
    for path in sorted(directory.rglob("*")):
        if path.is_file():
            manifest_rows.append({
                "relative_path": path.relative_to(OUT).as_posix(),
                "size_bytes": path.stat().st_size,
                "sha256": sha256(path),
            })
manifest_csv = MANIFEST / f"phase7g_{PAIR_SLUG}_manifest.csv"
with manifest_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["relative_path", "size_bytes", "sha256"]
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
manifest_json = MANIFEST / f"phase7g_{PAIR_SLUG}_manifest.json"
write_json(manifest_json, {
    "schema_version": "phase7g_pair_manifest_v1",
    "created_at_utc": now(),
    "rows": manifest_rows,
})
sha_file = MANIFEST / "SHA256SUMS"
sha_file.write_text(
    "".join(
        f"{row['sha256']}  {row['relative_path']}\n"
        for row in manifest_rows
    ),
    encoding="utf-8",
)
for row in manifest_rows:
    path = resolve_inside(OUT, row["relative_path"])
    assert path.stat().st_size == int(row["size_bytes"])
    assert sha256(path) == row["sha256"]

marker = {
    "schema_version": (
        "phase7g_pair_ort_mobile_export_preflight_complete_v1"
        if PHASE7G_PASSED
        else "phase7g_pair_ort_mobile_export_preflight_blocked_v1"
    ),
    "created_at_utc": now(),
    "status": "complete" if PHASE7G_PASSED else "technical_blocked",
    "decision": (
        "ort_mobile_export_and_int8_compatibility_passed_phase7h_authorized"
        if PHASE7G_PASSED
        else "phase7g_requires_technical_repair"
    ),
    "pair": PAIR,
    "student_id": STUDENT_ID,
    "seed": SEED,
    "strategy": LOCKED["strategy"],
    "modern_onnx_exports_complete": EXPORTS_COMPLETE,
    "fp32_ort_parity_passed": FP32_PARITY_PASSED,
    "ort_int8_compatibility_passed": ORT_INT8_PREFLIGHT_PASSED,
    "caption_quality_evaluated": False,
    "physical_android_benchmark_performed": False,
    "new_training_performed": False,
    "checkpoint_reselection_performed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "manifest": {
        "csv": {
            "relative_path": manifest_csv.relative_to(OUT).as_posix(),
            "size_bytes": manifest_csv.stat().st_size,
            "sha256": sha256(manifest_csv),
            "rows": len(manifest_rows),
        },
        "json": {
            "relative_path": manifest_json.relative_to(OUT).as_posix(),
            "size_bytes": manifest_json.stat().st_size,
            "sha256": sha256(manifest_json),
        },
        "sha256sums": {
            "relative_path": sha_file.relative_to(OUT).as_posix(),
            "size_bytes": sha_file.stat().st_size,
            "sha256": sha256(sha_file),
        },
    },
    "manifest_rows": len(manifest_rows),
    "elapsed_seconds_cell06": time.perf_counter() - STARTED,
}
marker_name = (
    f"PHASE7G_{MARKER_BASE}_ORT_MOBILE_EXPORT_PREFLIGHT_COMPLETE.json"
    if PHASE7G_PASSED
    else f"PHASE7G_{MARKER_BASE}_ORT_MOBILE_EXPORT_PREFLIGHT_TECHNICAL_BLOCKED.json"
)
write_json(OUT / marker_name, marker)

print("PHASE 07G CELL 6 - FINAL")
print("Status                  :", marker["status"].upper())
print("Pair                    :", PAIR)
print("Modern ONNX export      :", EXPORTS_COMPLETE)
print("FP32 ORT parity         :", FP32_PARITY_PASSED)
print("ORT INT8 preflight      :", ORT_INT8_PREFLIGHT_PASSED)
print("Caption quality         : NOT YET EVALUATED")
print("Physical Android        : NOT BENCHMARKED")
print("New training            : NOT PERFORMED")
print("Held-out Test           : SEALED / NOT USED")
print("Manifest                :", len(manifest_rows), "/", len(manifest_rows), "VERIFIED")
print("Next                    :", "Phase 07H full ONNX quality gate" if PHASE7G_PASSED else "repair Phase 07G technical blocker")
print("Progress                : 6 / 6 code cells")


PHASE 07G CELL 6 - FINAL
Status                  : COMPLETE
Pair                    : T1-S1
Modern ONNX export      : True
FP32 ORT parity         : True
ORT INT8 preflight      : True
Caption quality         : NOT YET EVALUATED
Physical Android        : NOT BENCHMARKED
New training            : NOT PERFORMED
Held-out Test           : SEALED / NOT USED
Manifest                : 24 / 24 VERIFIED
Next                    : Phase 07H full ONNX quality gate
Progress                : 6 / 6 code cells
